In [9]:
import pandas as pd
import sqlite3

# Load cleaned dataset
df = pd.read_csv("advertising_data_cleaned.csv")

# Create a SQLite database
conn = sqlite3.connect("advertising_analysis.db")

# Create a SQL table
df.to_sql(
    "advertising_data",
    conn,
    if_exists="replace",
    index=False
)

print("SQL database and table created successfully!")

SQL database and table created successfully!


#### How can the company optimize advertising spend by identifying which advertising channels, platforms, and creative formats deliver the highest click-through rates at the lowest cost, while staying within budget?

In [16]:
query = "SELECT * FROM advertising_data LIMIT 10;"
pd.read_sql_query(query, conn)

,campaign_item_id,no_of_days,time,ext_service_id,ext_service_name,creative_id,creative_width,creative_height,search_tags,template_id,...,clicks,stats_currency,currency_code,exchange_rate,media_cost_usd,search_tag_cat,cmi_currency_code,timezone,weekday_cat,keywords
0,2733,7,2022-05-01,128,Facebook Ads,1000,300.0,250.0,#The Power of X,90.0,...,8,SGD,SGD,1,14.058514,Others,SGD,Asia/Singapore,week_end,delicate bracelets
1,2733,8,2022-05-02,16,DV360,1000,300.0,250.0,#Be Bold. Be X,90.0,...,44,SGD,SGD,1,99.633496,Others,SGD,Asia/Singapore,week_day,summer jewelry
2,2733,9,2022-05-03,128,Facebook Ads,1000,300.0,250.0,#Embrace Your Individuality with X,90.0,...,32,SGD,SGD,1,109.419677,Others,SGD,Asia/Singapore,week_day,artisan jewelry
3,2733,10,2022-05-04,128,Facebook Ads,1000,300.0,250.0,#Be Bold. Be X,90.0,...,48,SGD,SGD,1,115.209499,Others,SGD,Asia/Singapore,week_day,layered bracelets
4,2733,11,2022-05-05,4,Google Ads,1000,300.0,250.0,#Be Bold. Be X,90.0,...,20,SGD,SGD,1,66.990104,Others,SGD,Asia/Singapore,week_day,minimalist jewelry
5,2766,53,2022-05-31,4,Google Ads,1005,0.0,0.0,#Be Bold. Be X,23.0,...,11,USD,USD,1,35.673124,Youtube,USD,America/New_York,week_day,beaded bracelets
6,2766,54,2022-06-01,4,Google Ads,1005,0.0,0.0,#The Ultimate Fashion Statement with X,23.0,...,18,USD,USD,1,43.150656,Youtube,USD,America/New_York,week_day,bridal jewelry
7,2766,55,2022-06-02,16,DV360,1005,0.0,0.0,#The Ultimate Fashion Statement with X,23.0,...,12,USD,USD,1,31.436393,Youtube,USD,America/New_York,week_day,unique and trendy jewelry
8,2766,56,2022-06-03,16,DV360,1005,0.0,0.0,#The X Factor - Fashion for the Fearless,23.0,...,5,USD,USD,1,5.409166,Youtube,USD,America/New_York,week_day,clip-on earrings
9,2766,57,2022-06-04,128,Facebook Ads,1005,0.0,0.0,#Be Bold. Be X,23.0,...,8,USD,USD,1,5.965384,Youtube,USD,America/New_York,week_day,bohemian jewelry


In [13]:
query = """
SELECT
    channel_name,
    ext_service_name AS advertising_platform,
    creative_width,
    creative_height,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(media_cost_usd), 2) AS total_spend_usd,

    ROUND(
        100.0 * SUM(clicks) / NULLIF(SUM(impressions), 0),
        2
    ) AS ctr_pct,

    ROUND(
        SUM(media_cost_usd) / NULLIF(SUM(clicks), 0),
        4
    ) AS cpc_usd,

    ROUND(
        1000.0 * SUM(media_cost_usd)
        / NULLIF(SUM(impressions), 0),
        2
    ) AS cpm_usd

FROM advertising_data

WHERE impressions > 0
  AND clicks >= 0
  AND clicks <= impressions
  AND media_cost_usd >= 0

GROUP BY
    channel_name,
    ext_service_name,
    creative_width,
    creative_height

HAVING SUM(impressions) >= 1000
   AND SUM(clicks) >= 30

ORDER BY
    ctr_pct DESC,
    cpc_usd ASC;
"""

pd.read_sql_query(query, conn)


,channel_name,advertising_platform,creative_width,creative_height,total_impressions,total_clicks,total_spend_usd,ctr_pct,cpc_usd,cpm_usd
0,Social,DV360,NaN,NaN,1608443,128530,19737.54,7.99,0.1536,12.27
1,Mobile,Google Ads,NaN,NaN,1190918,92889,14463.82,7.80,0.1557,12.15
2,Display,Facebook Ads,NaN,NaN,1335101,102257,13613.26,7.66,0.1331,10.20
3,Video,Facebook Ads,NaN,NaN,1394144,106069,15416.36,7.61,0.1453,11.06
4,Search,Facebook Ads,NaN,NaN,1481104,111488,18640.06,7.53,0.1672,12.59
5,Video,Google Ads,NaN,NaN,1401894,104091,17059.55,7.43,0.1639,12.17
6,Social,Google Ads,NaN,NaN,1283902,92913,13906.29,7.24,0.1497,10.83
7,Display,DV360,NaN,NaN,1616708,116095,19553.01,7.18,0.1684,12.09
8,Mobile,DV360,NaN,NaN,1217370,82538,13527.24,6.78,0.1639,11.11
9,Search,DV360,NaN,NaN,1296671,85727,13943.03,6.61,0.1626,10.75
